In [1]:
import numpy as np 
import pandas as pd 

In [2]:
data=pd.read_csv("Data/Wine.csv")

In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1990 entries, 0 to 1989
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Unnamed: 0   1990 non-null   int64 
 1   winery       1990 non-null   object
 2   country      1988 non-null   object
 3   points       1990 non-null   int64 
 4   description  1990 non-null   object
 5   designation  1491 non-null   object
dtypes: int64(2), object(4)
memory usage: 93.4+ KB


In [4]:
data.drop('Unnamed: 0',inplace=True,axis=1)

In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1990 entries, 0 to 1989
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   winery       1990 non-null   object
 1   country      1988 non-null   object
 2   points       1990 non-null   int64 
 3   description  1990 non-null   object
 4   designation  1491 non-null   object
dtypes: int64(1), object(4)
memory usage: 77.9+ KB


In [6]:
data.describe()

,points
count,1990.000000
mean,89.452261
std,2.354565
min,83.000000
25%,88.000000
50%,90.000000
75%,91.000000
max,96.000000


In [7]:
data.head()

,winery,country,points,description,designation
0,Heitz,US,96,This tremendous 100% varietal wine hails from ...,Martha's Vineyard
1,Bodega Carmen Rodríguez,Spain,96,"Ripe aromas of fig, blackberry and cassis are ...",Carodorum Selección Especial Reserva
2,Macauley,US,96,Mac Watson honors the memory of a wine once ma...,Special Selected Late Harvest
3,Ponzi,US,96,"This spent 20 months in 30% new French oak, an...",Reserve
4,Domaine de la Bégude,France,95,"This is the top wine from La Bégude, named aft...",La Brûlade


In [8]:
data.isna().sum()

winery           0
country          2
points           0
description      0
designation    499
dtype: int64

In [9]:
data.shape

(1990, 5)

In [10]:
data['label']=(data['points']>=90).astype(int)

In [11]:
data.head()

,winery,country,points,description,designation,label
0,Heitz,US,96,This tremendous 100% varietal wine hails from ...,Martha's Vineyard,1
1,Bodega Carmen Rodríguez,Spain,96,"Ripe aromas of fig, blackberry and cassis are ...",Carodorum Selección Especial Reserva,1
2,Macauley,US,96,Mac Watson honors the memory of a wine once ma...,Special Selected Late Harvest,1
3,Ponzi,US,96,"This spent 20 months in 30% new French oak, an...",Reserve,1
4,Domaine de la Bégude,France,95,"This is the top wine from La Bégude, named aft...",La Brûlade,1


In [12]:
data['label'].value_counts()

label
1    1191
0     799
Name: count, dtype: int64

In [13]:
from sklearn.model_selection import train_test_split

In [14]:
X_train,X_test,y_train,y_test=train_test_split(data['description'],data['label'],test_size=0.2,random_state=42)

In [15]:
print(len(X_train), len(y_train))
print(len(X_test), len(y_test))

1592 1592
398 398


In [16]:
from sklearn.pipeline import Pipeline 
from sklearn.linear_model import LogisticRegression 
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer 
from sklearn.metrics import classification_report,f1_score,accuracy_score,confusion_matrix
import os

In [34]:
def evaluate_model(name,model,Xtrain,Xtest,ytrain,ytest): 
    model.fit(Xtrain,ytrain)
    ypred=model.predict(Xtest)
    print(f"\n {name}")
    print(f"\n classification_report \n {classification_report(ytest,ypred)}")
    print(f"accuracy_score {accuracy_score(ytest,ypred)}")
    print(f"f1_score {f1_score(ytest,ypred)}")
    print(f"confusion_matrix \n {confusion_matrix(ytest,ypred)}")

In [40]:
svc_pipeline=Pipeline([
                         ('countvect',CountVectorizer(lowercase=True,stop_words='english',ngram_range=(1,1),min_df=2 if len(data)>200 else 1)),
                         ('SVC',LinearSVC())
                        ])
loguni_pipeline=Pipeline([
                         ('countvect',CountVectorizer(lowercase=True,stop_words='english',ngram_range=(1,1),min_df=2 if len(data)>200 else 1)),
                         ('Logistic',LogisticRegression(max_iter=200,class_weight='balanced',solver='liblinear'))
                        ])
logbi_pipeline=Pipeline([
                         ('countvect',CountVectorizer(lowercase=True,stop_words='english',ngram_range=(1,2),min_df=2 if len(data)>200 else 1)),
                         ('Logistic',LogisticRegression(max_iter=200,class_weight='balanced',solver='liblinear'))
                        ])


In [41]:

evaluate_model('Logistic Unigram', loguni_pipeline, X_train, X_test, y_train, y_test)


 Logistic Unigram

 classification_report 
               precision    recall  f1-score   support

           0       0.70      0.72      0.71       162
           1       0.81      0.79      0.80       236

    accuracy                           0.76       398
   macro avg       0.75      0.76      0.75       398
weighted avg       0.76      0.76      0.76       398

accuracy_score 0.7613065326633166
f1_score 0.7965738758029979
confusion_matrix 
 [[117  45]
 [ 50 186]]


In [42]:
evaluate_model('Logistic Bigram', logbi_pipeline, X_train, X_test, y_train, y_test)


 Logistic Bigram

 classification_report 
               precision    recall  f1-score   support

           0       0.74      0.76      0.75       162
           1       0.83      0.81      0.82       236

    accuracy                           0.79       398
   macro avg       0.78      0.79      0.78       398
weighted avg       0.79      0.79      0.79       398

accuracy_score 0.7914572864321608
f1_score 0.8222698072805139
confusion_matrix 
 [[123  39]
 [ 44 192]]


In [43]:
logtfidf_pipeline=Pipeline([
                         ('vect',TfidfVectorizer(lowercase=True,stop_words='english',ngram_range=(1,2),min_df=3 if len(data)>200 else 1,sublinear_tf=True)),
                         ('Logistic',LogisticRegression(max_iter=200,class_weight='balanced',solver='liblinear'))
                        ])

In [48]:
evaluate_model('Logistic TFidf', logtfidf_pipeline, X_train, X_test, y_train, y_test)


 Logistic TFidf

 classification_report 
               precision    recall  f1-score   support

           0       0.73      0.72      0.72       162
           1       0.81      0.81      0.81       236

    accuracy                           0.78       398
   macro avg       0.77      0.77      0.77       398
weighted avg       0.78      0.78      0.78       398

accuracy_score 0.7763819095477387
f1_score 0.8118393234672304
confusion_matrix 
 [[117  45]
 [ 44 192]]


In [46]:
svmtfidf_pipeline=Pipeline([
                         ('vect',TfidfVectorizer(lowercase=True,stop_words='english',ngram_range=(1,2),min_df=3 if len(data)>200 else 1,sublinear_tf=True)),
                         ('SVM',LinearSVC())
                        ])

In [47]:
evaluate_model('SVM TFidf', svmtfidf_pipeline, X_train, X_test, y_train, y_test)


 SVM TFidf

 classification_report 
               precision    recall  f1-score   support

           0       0.79      0.68      0.73       162
           1       0.80      0.87      0.83       236

    accuracy                           0.79       398
   macro avg       0.79      0.78      0.78       398
weighted avg       0.79      0.79      0.79       398

accuracy_score 0.7939698492462312
f1_score 0.8340080971659919
confusion_matrix 
 [[110  52]
 [ 30 206]]
